# Sayari MCP — client setup & credential test

A self-contained walkthrough of connecting to the **Sayari MCP server** and handing its tools to an LLM.

Use it to (1) verify a set of client credentials works, and (2) copy the connection pattern into your own app.

**What happens**

1. Exchange `client_id` / `client_secret` for an Auth0 access token (OAuth 2.0 client-credentials grant).
2. Open an MCP session over Streamable HTTP, sending that token as a `Bearer` header.
3. List the tools the server exposes and call one directly.
4. Hand those tools to an LLM (via [LiteLLM](https://docs.litellm.ai/), so any provider works) and let it run a short agent loop.

Steps 1–3 are the credential test — if you only need to confirm access works, you can stop after step 3.

**What you need**

- Python 3.10+
- A `.env` file in this notebook's folder (or any parent directory) containing your Sayari credentials:

  ```
  AUTH0_CLIENT_ID=...
  AUTH0_CLIENT_SECRET=...
  ```

- (Only for step 4) an API key for whichever LLM provider you point LiteLLM at.

Everything else — URLs, audience, model name — is a plain variable in the configuration cell below, so you can point this at a different environment by editing one place.

> Runs on either major version of the MCP Python SDK (v1.x or v2.x). The two differ in a few
> details; the configuration cell detects which you have and adapts. See *MCP SDK versions* at the bottom.

In [ ]:
%pip install --quiet "mcp>=1.9" "litellm>=1.60" "python-dotenv>=1.0"
# If this installs or upgrades anything, restart the kernel before continuing.

## 0. Configuration

Only the two secrets come from `.env`. Everything else is set here.

In [ ]:
import json
import os
from contextlib import asynccontextmanager
from importlib.metadata import version as _pkg_version

from dotenv import find_dotenv, load_dotenv

# --- Sayari MCP endpoint -------------------------------------------------
SAYARI_MCP_URL = "https://mcp.sayari.com/mcp"

# --- Auth0 client-credentials flow ---------------------------------------
AUTH0_TOKEN_URL = "https://sayari.auth0.com/oauth/token"
AUTH0_AUDIENCE = "https://mcp.sayari.com/"  # the trailing slash matters

# --- LLM (step 4 only; any LiteLLM-supported model string) ---------------
LLM_MODEL = "anthropic/claude-sonnet-5"
LLM_API_KEY_ENV = "ANTHROPIC_API_KEY"  # env var LiteLLM reads for the above provider
MAX_TOOL_RESULT_CHARS = 20_000  # truncate large tool payloads before feeding them back

# --- MCP SDK compatibility -----------------------------------------------
# v1.x is built on `httpx`, v2.x on its successor `httpx2`. Pick the one our SDK expects.
MCP_VERSION = _pkg_version("mcp")
MCP_MAJOR = int(MCP_VERSION.split(".")[0])
if MCP_MAJOR >= 2:
    import httpx2 as http_lib
else:
    import httpx as http_lib


def field(obj, *names):
    """Read a field the MCP SDK spells camelCase in v1 and snake_case in v2."""
    for name in names:
        if hasattr(obj, name):
            return getattr(obj, name)
    raise AttributeError(f"{type(obj).__name__} has none of {names}")


# --- Secrets -------------------------------------------------------------
# override=True so the file always wins: a stale `export AUTH0_CLIENT_ID=...` left in
# the shell would otherwise silently shadow .env and you'd test the wrong credentials.
DOTENV_PATH = find_dotenv(usecwd=True)
if not DOTENV_PATH:
    raise FileNotFoundError("No .env found in this folder or a parent — see the cell above.")
load_dotenv(DOTENV_PATH, override=True)

CLIENT_ID = os.environ["AUTH0_CLIENT_ID"]
CLIENT_SECRET = os.environ["AUTH0_CLIENT_SECRET"]

print(f"mcp SDK  : {MCP_VERSION} (using {http_lib.__name__})")
print(f".env     : {DOTENV_PATH}")
print(f"client_id: {CLIENT_ID[:6]}…{CLIENT_ID[-4:]}  (secret loaded: {bool(CLIENT_SECRET)})")

## 1. Get an access token

This is the credential test. A token here means the credentials are valid; a `401`/`403` means they are not (see *Troubleshooting* at the bottom).

Tokens are long-lived (24h) but not permanent, so we cache one and reuse it for the rest of the notebook. A production client should do the same and refresh shortly before `expires_in` elapses.

**To test a different set of credentials:** edit `.env`, then re-run the configuration cell and this one. The `.env` file always wins over shell variables, and the token cache is keyed to the credentials that produced it, so nothing stale carries over.

In [ ]:
_token_cache = None
_token_cache_key = None


def get_access_token(refresh: bool = False) -> str:
    """Exchange client credentials for an Auth0 access token scoped to the MCP audience.

    Cached per credential set, so editing .env and re-running the config cell
    transparently fetches a new token instead of reusing the previous one.
    """
    global _token_cache, _token_cache_key
    cache_key = (CLIENT_ID, CLIENT_SECRET, AUTH0_AUDIENCE)
    if _token_cache and _token_cache_key == cache_key and not refresh:
        return _token_cache

    response = http_lib.post(
        AUTH0_TOKEN_URL,
        json={
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "audience": AUTH0_AUDIENCE,
            "grant_type": "client_credentials",
        },
        headers={"accept": "application/json"},
        timeout=30.0,
    )
    if response.is_error:
        raise RuntimeError(f"Auth0 token request failed ({response.status_code}): {response.text}")

    payload = response.json()
    print(f"✅ token acquired — type={payload['token_type']}, expires_in={payload['expires_in']}s")
    _token_cache, _token_cache_key = payload["access_token"], cache_key
    return _token_cache


get_access_token()
None  # don't echo the token

## 2. Open an MCP session

The Sayari MCP server speaks **Streamable HTTP**. Authentication is just an `Authorization: Bearer <token>` header on that transport, which you set by handing the SDK your own pre-configured async HTTP client.

We wrap the three nested context managers (HTTP client → transport → session) in one helper so every cell below can just say:

```python
async with sayari_session() as session:
    ...
```

Opening and closing per cell is deliberate: MCP sessions are built on `anyio` task groups, which must be entered and exited in the **same** task. A notebook runs each `await` cell in its own task, so a session held open across cells would blow up on close. This is also exactly the shape you want in application code.

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

last_init_result = None


@asynccontextmanager
async def sayari_session():
    """Yield an initialized MCP ClientSession authenticated against Sayari."""
    global last_init_result
    headers = {"Authorization": f"Bearer {get_access_token()}"}
    async with http_lib.AsyncClient(headers=headers, timeout=120.0) as http_client:
        async with streamable_http_client(SAYARI_MCP_URL, http_client=http_client) as streams:
            # v1 yields (read, write, get_session_id); v2 yields (read, write).
            read, write = streams[0], streams[1]
            async with ClientSession(read, write) as session:
                last_init_result = await session.initialize()
                yield session


async with sayari_session() as session:
    info = field(last_init_result, "server_info", "serverInfo")
    print(f"✅ connected to {info.name} v{info.version}")

### What tools does the server expose?

In [ ]:
async with sayari_session() as session:
    mcp_tools = (await session.list_tools()).tools

print(f"{len(mcp_tools)} tools available:\n")
for tool in mcp_tools:
    summary = (tool.description or "").split("\n")[0][:100]
    print(f"  {tool.name:<28} {summary}")

In [ ]:
# A tool's input schema is standard JSON Schema — this is what you hand to an LLM.
search_tool = next(t for t in mcp_tools if t.name == "search_entities")
print(json.dumps(field(search_tool, "input_schema", "inputSchema"), indent=2))

## 3. Call a tool directly

No LLM involved — this proves the token is accepted for actual data access, not just for opening the connection.

In [ ]:
def result_to_text(result) -> str:
    """Flatten an MCP tool result into plain text (Sayari tools return JSON as text blocks)."""
    return "\n".join(block.text for block in result.content if getattr(block, "type", None) == "text")


async with sayari_session() as session:
    result = await session.call_tool("search_entities", {"query": "Sinopec", "limit": 3})

print(f"is_error: {field(result, 'is_error', 'isError')}\n")
print(result_to_text(result)[:2000])

## 4. Hand the tools to an LLM

MCP tool definitions map 1:1 onto the OpenAI-style function-calling format that LiteLLM normalises across providers, so wiring them up is a small translation plus a loop:

> ask the model → if it requested tools, call them over MCP and append the results → ask again → repeat until it answers.

Set the provider key first, e.g. `export ANTHROPIC_API_KEY=...` (or add it to `.env`, since `load_dotenv()` above puts it in the environment). To use a different provider, change `LLM_MODEL` and `LLM_API_KEY_ENV` in the configuration cell — nothing else changes.

In [ ]:
def to_openai_tools(tools) -> list[dict]:
    """MCP tool definitions -> OpenAI-style function schemas."""
    return [
        {
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description or "",
                "parameters": field(tool, "input_schema", "inputSchema"),
            },
        }
        for tool in tools
    ]


openai_tools = to_openai_tools(mcp_tools)
print(f"{len(openai_tools)} tools converted — e.g.\n")
print(json.dumps(openai_tools[7], indent=2)[:600])

In [ ]:
import litellm


async def run_agent(question: str, max_turns: int = 6, verbose: bool = True) -> str:
    """Minimal tool-calling loop: the LLM decides, we execute against the MCP session."""
    messages: list[dict] = [{"role": "user", "content": question}]

    async with sayari_session() as session:
        for turn in range(1, max_turns + 1):
            response = await litellm.acompletion(
                model=LLM_MODEL,
                messages=messages,
                tools=openai_tools,
            )
            message = response.choices[0].message
            messages.append(message.model_dump(exclude_none=True))

            if not message.tool_calls:
                return message.content

            for call in message.tool_calls:
                arguments = json.loads(call.function.arguments or "{}")
                if verbose:
                    print(f"[turn {turn}] → {call.function.name}({json.dumps(arguments)[:120]})")

                tool_result = await session.call_tool(call.function.name, arguments)
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": call.id,
                        "content": result_to_text(tool_result)[:MAX_TOOL_RESULT_CHARS] or "(empty result)",
                    }
                )

    return f"(stopped: reached max_turns={max_turns})"

In [ ]:
assert os.environ.get(LLM_API_KEY_ENV), f"Set {LLM_API_KEY_ENV} to run this cell"

answer = await run_agent(
    "Search Sayari for the company 'Sinopec Chemical Commercial International'. "
    "Report its entity id and type, then check it against watchlists and summarise what you find."
)
print("\n" + answer)

## Troubleshooting

| Symptom | Likely cause |
| --- | --- |
| `401 access_denied` from Auth0 | Wrong `client_id` / `client_secret`. |
| `403 access_denied` — *"not authorized to access resource server … You need to create a `client-grant`"* | The credentials are real, but this Auth0 client has no grant for the **MCP** API. Credentials scoped only to the Sayari REST API (`sayari.com` audience) fail here — MCP access must be granted separately. |
| Token succeeds, but the MCP connection returns `401` | Token issued for a different audience — `AUTH0_AUDIENCE` must be `https://mcp.sayari.com/`, trailing slash included. |
| `403` on one specific tool call, others fine | Credentials are valid for MCP but the account lacks entitlement for that dataset. |
| Credentials look wrong / you changed `.env` and nothing changed | The configuration cell prints the `.env` path and the `client_id` actually in use — check it's the file you edited. `.env` overrides shell variables, so a stale `export` won't shadow it. |
| `KeyError: 'AUTH0_CLIENT_ID'` | `.env` found but the key is missing or misspelled. |
| `FileNotFoundError: No .env found` | `.env` must sit in this notebook's folder or a parent. |
| `NameError` / `ImportError` on a name that exists in the file | The kernel is running stale cells, or `%pip install` changed a package mid-session. Restart the kernel and run from the top. |
| `RuntimeError: Attempted to exit cancel scope in a different task` | An MCP session was opened in one cell and closed in another. Keep each `async with sayari_session()` block inside a single cell. |

### MCP SDK versions

The SDK changed in v2 in ways that break naive copy-paste between versions. This notebook handles all of it
(`MCP_MAJOR` picks the HTTP library, `field()` reads either spelling, and the transport result is
indexed rather than unpacked), but if you're porting the code elsewhere, these are the differences:

| | v1.x | v2.x |
| --- | --- | --- |
| HTTP library | `httpx` | `httpx2` |
| transport yields | `(read, write, get_session_id)` | `(read, write)` |
| field names | `serverInfo`, `inputSchema`, `isError` | `server_info`, `input_schema`, `is_error` |
| legacy alias | `streamablehttp_client(url, headers={...})` | removed |

`streamable_http_client(url, http_client=...)` — the form used here — exists in both, which is why this
notebook uses it rather than the older `streamablehttp_client`. The transport, the token, and the
`Bearer` header are identical either way.

### Porting this to application code

The `sayari_session()` helper above is already the production shape — copy it as-is:

```python
async def main():
    async with sayari_session() as session:
        tools = (await session.list_tools()).tools
        result = await session.call_tool("search_entities", {"query": "..."})
```

If you're targeting a single known SDK version, drop the compatibility shims and inline the values for
that version.

Higher-level frameworks wrap the same two pieces — token in a header, Streamable HTTP transport. For example with pydantic-ai:

```python
from pydantic_ai.mcp import MCPServerStreamableHTTP

toolset = MCPServerStreamableHTTP(
    url=SAYARI_MCP_URL,
    headers={"Authorization": f"Bearer {get_access_token()}"},
)
```